# **더 알아보기 : 내가 쓴 숫자를 분류할 수 있을까?**


 내가 직접 쓴 숫자를 MNIST 신경망 모델로 분류해 봅시다.


## **1. 숫자 쓰기**

(1) 흰 종이에 검은색 펜으로 원하는 숫자를 작성해 본다.
    (두꺼운 검은펜을 사용한다. )


(2) 스마트폰으로 사진을 찍어 저장한다. (예 : 13.jpg)

## **2. MNIST 신경망 모델 파일 준비하기**

실습활동에서 저장한 MNIST 신경망 모델 파일을 준비한다. mnist_model.h5

In [ ]:
from tensorflow import keras
new_model = keras.models.load_model('mnist_model.h5')

3. 구글 코랩 파일에 숫자와 신경망 모델 파일 업로드하기
(1) 크롬 브라우저에서 구글 계정으로 로그인한다.

**이미지를 읽고, 회색조로 변환한 후, 화면에 표시함 **

In [ ]:
import cv2
import matplotlib.pyplot as plt

# 이미지 파일 '13.jpg'을 읽어 들임
image = cv2.imread('13.png')

# 이미지의 크기를 출력
print("이미지 크기 ", image.shape)

# 이미지를 회색조로 변환하여 출력
plt.imshow(image, cmap = plt.cm.gray)
plt.show()

# 이미지를 복사한 후 회색조로 변환하여 출력
grey = cv2.cvtColor(image.copy(), cv2.COLOR_BGR2GRAY)
plt.imshow(grey, cmap = plt.cm.gray)
plt.show()

**숫자 인식하기**

1. 임계값: 임계값 70을 사용하여 회색조 이미지를 이진 이미지로 변환합니다.

2. 윤곽선 감지: cv2.findContours() 함수를 사용하여 이진 이미지의 윤곽선을 식별합니다.

3. 숫자 감지: 작은 윤곽선을 필터링하고 잠재적인 숫자 주위의 경계 상자를 추출합니다.

4. 전처리: 각 숫자 이미지의 크기를 조정하고, 채우고, 목록에 저장합니다.

5. 시각화: 경계 상자와 전처리된 각 개별 숫자가 포함된 원본 이미지를 표시합니다.

In [ ]:
import numpy as np

# 1. 회색조 이미지를 이진 이미지로 변환
ret, thresh = cv2.threshold(grey.copy(), 70, 255, cv2.THRESH_BINARY_INV)

# 2. 이진 이미지에서 윤곽을 찾음
contours, hierarchy = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 3. 윤곽 개수 출력
print("윤곽 개수:", len(contours))

# 4. 전처리된 숫자 저장용 빈 리스트 초기화
preprocessed_digits = []

# 5. 숫자 개수 카운터 초기화
countImage = 0

# 6. 각 윤곽에 대해 반복
for c in contours:
    # 7. 각 윤곽의 경계 상자 정보를 추출
    x, y, w, h = cv2.boundingRect(c)

    # 8. 작은 윤곽(노이즈 방지) 필터링
    if w > 1 and h > 1:
        # 9. 경계 상자 좌표 출력
        print("경계 상자:", x, y, w, h)

        # 10. 숫자 개수 증가
        countImage += 1

        # 11. 탐지된 숫자 주위에 사각형 그리기
        cv2.rectangle(image, (x, y), (x + w, y + h), color=(255, 0, 0), thickness=2)

        # 12. 이진 이미지에서 숫자 영역 추출
        digit = thresh[y:y + h, x:x + w]

        # 13. 숫자 이미지 크기 조정
        s = 20 / max(digit.shape)                      # 긴 변을 20으로 맞출 배율
        nh, nw = max(1, round(digit.shape[0]*s)), max(1, round(digit.shape[1]*s))
        resized_digit = cv2.resize(digit, (nw, nh))    # 비율 유지
        py, px = (28-nh)//2, (28-nw)//2                # 28x28 중앙에 배치
        # 14. 크기 조정된 숫자 이미지에 0으로 패딩
        padded_digit = np.pad(resized_digit, ((py, 28-nh-py), (px, 28-nw-px)), "constant", constant_values=0)

        # 15. 패딩된 숫자 이미지 모양 출력
        #print("패딩된 숫자 모양:", padded_digit.shape)

        # 16. 전처리된 숫자 리스트에 추가
        preprocessed_digits.append(padded_digit)

# 17. 탐지된 숫자 개수 출력
print("총 숫자:", countImage)

# 18. 그래프 크기 조정 (가시성 향상)
plt.rcParams['figure.figsize'] = (4, 4)

# 19. 원본 이미지에 경계 상자 표시하기
plt.imshow(image, cmap=plt.cm.gray)
plt.show()

# 20. 전처리된 숫자 리스트를 NumPy 배열로 변환하기
inp = np.array(preprocessed_digits)

# 21. 각 전처리된 숫자 개별로 보여주기
for i in range(len(inp)):
    plt.imshow(inp[i], cmap='gray', interpolation='none')
    plt.show()


In [ ]:
for i, digit in enumerate(preprocessed_digits):
    prediction = new_model.predict(digit.reshape(1, 28,28))
    plt.imshow(digit.reshape(28, 28), cmap="gray")
    plt.title("Predict {}".format(np.argmax(prediction)))
    plt.show()